# 14 - Local Knowledge Retrieval with SochDB

This notebook focuses on SochDB's strongest current wedge:

- local knowledge retrieval
- lightweight RAG foundations
- Python-first embedded evaluation

It is designed for evaluators who want to answer one narrow question:

> Is SochDB a credible and simpler local retrieval workflow compared with stitching together multiple local tools?

This notebook shows:

1. storing documents in SochDB
2. generating deterministic local embeddings
3. indexing and retrieving with SochDB's HNSW index
4. comparing a fast preset and a quality preset
5. fetching full records back from the database


## 1. Setup

Recommended install:

```bash
pip install sochdb numpy
```

This notebook stays fully local and uses deterministic embeddings so the entire workflow is reproducible without external model APIs.

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
from sochdb import Database, HnswIndex


In [ ]:
DB_PATH = Path("./local_retrieval_demo_db")
DIMENSION = 64

DOCUMENTS = [
    {
        "id": 201,
        "title": "Laptop VPN Setup",
        "body": "To access internal dashboards, install the company VPN client and connect before opening private services.",
        "tags": ["it", "security", "access"],
    },
    {
        "id": 202,
        "title": "SSO Troubleshooting Guide",
        "body": "If SSO login fails, verify your device time, clear cached credentials, and retry before filing an access ticket.",
        "tags": ["it", "identity", "support"],
    },
    {
        "id": 203,
        "title": "Vendor Security Review Policy",
        "body": "New vendors handling customer or employee data must complete the security questionnaire and receive approval before purchase.",
        "tags": ["security", "vendor", "compliance"],
    },
    {
        "id": 204,
        "title": "Incident Rollback Checklist",
        "body": "If a deployment causes errors, stop rollout, restore the previous version, and confirm service health before resuming changes.",
        "tags": ["engineering", "incident", "deploy"],
    },
    {
        "id": 205,
        "title": "Expense Reimbursement Policy",
        "body": "Employees should submit receipts within 30 days and use the standard reimbursement form for travel and meals.",
        "tags": ["finance", "policy", "travel"],
    },
    {
        "id": 206,
        "title": "Access Review Checklist",
        "body": "Managers must review employee access to internal tools quarterly and remove permissions that are no longer required.",
        "tags": ["security", "audit", "access"],
    },
]

QUERIES = [
    "How do I access internal tools securely from my laptop?",
    "What should I do when a new vendor needs security approval?",
    "How do we recover after a bad deployment?",
]


def deterministic_embedding(text: str, dim: int = DIMENSION) -> np.ndarray:
    values = []
    counter = 0
    while len(values) < dim:
        digest = hashlib.sha256(f"{text}::{counter}".encode("utf-8")).digest()
        values.extend((byte / 255.0) * 2.0 - 1.0 for byte in digest)
        counter += 1
    vector = np.array(values[:dim], dtype=np.float32)
    norm = np.linalg.norm(vector)
    return vector if norm == 0 else vector / norm


if DB_PATH.exists():
    shutil.rmtree(DB_PATH)


## 2. Store the documents in SochDB

In this workflow, SochDB stores the full payloads. The vector index helps us retrieve the relevant IDs, and then we fetch the matching records back from the database.

In [ ]:
db = Database.open(str(DB_PATH))

vectors = []
ids = []

with db.transaction() as txn:
    for doc in DOCUMENTS:
        key = f"docs/{doc['id']}".encode("utf-8")
        db.put(key, json.dumps(doc).encode("utf-8"), txn.id)

        combined = f"{doc['title']}\n{doc['body']}\n{' '.join(doc['tags'])}"
        vectors.append(deterministic_embedding(combined))
        ids.append(doc['id'])

vectors = np.vstack(vectors).astype(np.float32)
ids = np.array(ids, dtype=np.uint64)

print(f"Stored {len(DOCUMENTS)} documents in {DB_PATH}")


## 3. Build two retrieval presets

These presets reflect the benchmark guidance we established for local retrieval evaluation:

- **fast**: good default local iteration path
- **quality**: better retrieval quality when you can spend more latency budget


In [ ]:
fast_index = HnswIndex(dimension=DIMENSION, m=16, ef_construction=100, precision="f32")
fast_index.add(vectors, ids)

quality_index = HnswIndex(dimension=DIMENSION, m=48, ef_construction=200, precision="f32")
quality_index.add(vectors, ids)

print("Fast preset   -> m=16, ef_construction=100, precision=f32")
print("Quality preset-> m=48, ef_construction=200, precision=f32")


## 4. Run local retrieval queries

Each query goes through the same flow:

1. embed the query locally
2. retrieve candidate IDs from the vector index
3. fetch the full matching records from SochDB


In [ ]:
def fetch_results(index: HnswIndex, query: str, k: int = 3):
    query_vec = deterministic_embedding(query)
    results = index.search(query_vec, k=k)
    rows = []
    for doc_id, score in results:
        payload = db.get(f"docs/{int(doc_id)}".encode("utf-8"))
        record = json.loads(payload.decode("utf-8"))
        rows.append({
            "id": record["id"],
            "title": record["title"],
            "score": round(float(score), 4),
        })
    return rows


for query in QUERIES:
    print("=" * 80)
    print("query:", query)
    print("fast preset")
    for row in fetch_results(fast_index, query):
        print("  ", row)
    print("quality preset")
    for row in fetch_results(quality_index, query):
        print("  ", row)


## 5. Inspect the database on disk

The local database is a directory. This matters because it makes the embedded path more concrete and easier to reason about during evaluation.

In [ ]:
sorted(str(path.relative_to(DB_PATH)) for path in DB_PATH.rglob("*"))[:20]


## 6. Why this wedge matters

This notebook does not try to prove that SochDB should replace every system.

It is showing a narrower product claim:

- if you want a Python-first local retrieval workflow
- if you want storage and retrieval to feel more unified
- if you want a clearer local evaluator path

then SochDB is worth evaluating seriously.

That is the strongest current entry point for the product.

## 7. Where to go next

After this notebook, useful next paths are:

- `13_sochdb_101.ipynb` for the broad front-door overview
- `6_transactions_kv.ipynb` for the database and transaction side
- `10_advanced_rag.ipynb` for more advanced retrieval flows
- the benchmark docs in the main SochDB repo for measured system comparisons


In [ ]:
db.close()
print("Closed database.")
